# Exploratory data analysis

Short EDA supporting the modelling choices in `credit_risk/`. It reuses the same
loader, target definition and split as `train.py`.

Set the path to `accepted_2007_to_2018Q4.csv` (Lending Club accepted loans
2007-2018Q4) in the `LENDING_CLUB_CSV` environment variable before starting
Jupyter. The data is not part of this repository.

Feature-vs-default relationships are shown on the **training period only**
(2013-01 to 2014-12) so the 2015 test set is not looked at.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from credit_risk.data import add_engineered_features, load_raw
from credit_risk.split import CALIBRATION_START, TEST_START, WINDOW_START, time_split
from credit_risk.target import TARGET, build_target

DATA = Path(os.environ["LENDING_CLUB_CSV"])
BLUE, ORANGE, MUTED = "#2a78d6", "#eb6834", "#898781"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "figure.figsize": (9, 4)})

In [ ]:
raw = add_engineered_features(load_raw(DATA))
print(f"{len(raw):,} loans, issued {raw.issue_d.min():%Y-%m} to {raw.issue_d.max():%Y-%m}")
raw["loan_status"].value_counts()

## 1. Why only matured loans with a final status

`Current` and late loans have no final outcome yet, and recent cohorts are
dominated by them. Treating them as non-defaults (as the original notebook did)
mislabels the target. The target keeps only `Fully Paid` vs `Charged Off`/`Default`
on loans whose term ended at least 3 months before the March 2019 snapshot.

In [ ]:
share_final = (raw.assign(year=raw.issue_d.dt.year)
               .groupby("year")["loan_status"]
               .apply(lambda s: s.isin(["Fully Paid", "Charged Off", "Default"]).mean()))
ax = share_final.plot.bar(color=BLUE, width=0.6)
ax.set(title="Share of loans with a final status, by issue year", xlabel="", ylabel="Share")
ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
plt.show()

## 2. Why the modelling window starts in 2013

Most credit-bureau attributes were not reported before mid-2012.

In [ ]:
bureau = ["tot_cur_bal", "mort_acc", "num_actv_rev_tl", "bc_util", "total_bal_ex_mort"]
missing = raw.groupby(raw.issue_d.dt.year)[bureau].apply(lambda d: d.isna().mean())
(missing * 100).round(0).astype(int).T

In [ ]:
df, counts = build_target(raw)
train, test = time_split(df)
print(counts)
print(f"train {len(train):,} loans, test {len(test):,} loans")

## 3. Default rate over time

The default rate moves between cohorts, which is why the split is by issue date
and why calibration is fitted on the latest part of the training period.

In [ ]:
window = df[df.issue_d >= WINDOW_START]
rate = window.groupby(window.issue_d.dt.to_period("Q"))[TARGET].mean()
ax = rate.plot(marker="o", color=BLUE)
for date, label in ((CALIBRATION_START, "calibration"), (TEST_START, "test")):
    ax.axvline(pd.Period(date, "Q"), color=MUTED, linewidth=1)
    ax.text(pd.Period(date, "Q"), rate.max(), f" {label}", color=MUTED, va="top")
ax.set(title="Default rate by issue quarter (matured loans)", xlabel="", ylabel="Default rate")
ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
plt.show()

## 4. Key application-time features vs default (training period only)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, col in zip(axes, ["int_rate", "fico_range_low", "dti"]):
    for value, color, label in ((0, BLUE, "Fully paid"), (1, ORANGE, "Default")):
        ax.hist(train.loc[train[TARGET] == value, col].dropna(), bins=40, density=True,
                histtype="step", linewidth=2, color=color, label=label)
    ax.set(title=col, yticks=[])
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
by_grade = train.groupby(train.sub_grade.str[0])[TARGET].agg(["mean", "size"])
ax = by_grade["mean"].plot.bar(color=BLUE, width=0.6, rot=0)
ax.set(title="Default rate by Lending Club grade (training period)", xlabel="Grade", ylabel="Default rate")
ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
plt.show()
by_grade

In [ ]:
by_purpose = train.groupby("purpose")[TARGET].agg(["mean", "size"]).sort_values("mean")
by_purpose[by_purpose["size"] >= 1000].style.format({"mean": "{:.1%}", "size": "{:,}"})